In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
import json
import random
import shutil

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import torchvision.transforms as transforms


PROJECT_DIR = Path("/content/drive/MyDrive/Underwater-Image-Data-set-main")

# New folder for this stage
DAY4_DIR = PROJECT_DIR / "Day_4_Preprocessing"

DAY4_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Day 4 directory:", DAY4_DIR)

Mounted at /content/drive
Project directory: /content/drive/MyDrive/Underwater-Image-Data-set-main
Day 4 directory: /content/drive/MyDrive/Underwater-Image-Data-set-main/Day_4_Preprocessing


In [3]:
PROJECT_DIR = Path("/content/drive/MyDrive/Underwater-Image-Data-set-main")
DAY4_DIR = PROJECT_DIR / "Day_4_Preprocessing"
DAY4_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
import re

pattern = re.compile(
    r"img_(\d+)_(input|target|gen)",
    re.IGNORECASE
)

records = []

image_extensions = [
    "*.png",
    "*.jpg",
    "*.jpeg",
    "*.bmp",
    "*.tif",
    "*.tiff"
]

for ext in image_extensions:
    for path in PROJECT_DIR.rglob(ext):

        # Don't accidentally read generated Day 4 files
        if DAY4_DIR in path.parents:
            continue

        match = pattern.fullmatch(path.stem)

        if match:
            image_id = int(match.group(1))
            role = match.group(2).lower()

            relative_parts = path.relative_to(PROJECT_DIR).parts

            source_archive = (
                relative_parts[0]
                if len(relative_parts) > 1
                else "unknown"
            )

            records.append({
                "image_id": image_id,
                "role": role,
                "file": str(path),
                "source_archive": source_archive
            })

df = pd.DataFrame(records)

print("Total role-labelled images:", len(df))
print()

print("Images by role:")
print(df["role"].value_counts())

print()

print("Unique image IDs:")
print(df["image_id"].nunique())

Total role-labelled images: 942

Images by role:
role
target    314
gen       314
input     314
Name: count, dtype: int64

Unique image IDs:
157


In [5]:
triplets = (
    df.pivot_table(
        index=["image_id", "source_archive"],
        columns="role",
        values="file",
        aggfunc="first"
    )
    .reset_index()
)

# Keep only complete triplets
triplets = triplets.dropna(
    subset=["input", "target", "gen"]
).reset_index(drop=True)

print("Complete triplets:", len(triplets))

print("\nColumns:")
print(triplets.columns.tolist())

triplets.head()

Complete triplets: 314

Columns:
['image_id', 'source_archive', 'gen', 'input', 'target']


role,image_id,source_archive,gen,input,target
0,1,epoch_0096,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...
1,1,extracted_dataset,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...
2,2,epoch_0096,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...
3,2,extracted_dataset,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...
4,3,epoch_0096,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...,/content/drive/MyDrive/Underwater-Image-Data-s...


In [6]:
from sklearn.model_selection import train_test_split

SEED = 42

# Make the split reproducible
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)


train_df, temp_df = train_test_split(
    triplets,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)


val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))
print("TOTAL:", len(train_df) + len(val_df) + len(test_df))

TRAIN: 219
VALIDATION: 47
TEST: 48
TOTAL: 314


In [9]:
from sklearn.model_selection import train_test_split

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# Create a UNIQUE sample ID
triplets["sample_id"] = (
    triplets["source_archive"].astype(str)
    + "_"
    + triplets["image_id"].astype(str)
)

print("Total triplets:", len(triplets))
print("Unique sample IDs:", triplets["sample_id"].nunique())

# Check that every sample_id is actually unique
assert triplets["sample_id"].nunique() == len(triplets)


train_df, temp_df = train_test_split(
    triplets,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)


val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))
print("TOTAL:", len(train_df) + len(val_df) + len(test_df))

Total triplets: 314
Unique sample IDs: 314

TRAIN: 219
VALIDATION: 47
TEST: 48
TOTAL: 314


In [10]:
train_df.to_csv(
    DAY4_DIR / "train_split.csv",
    index=False
)

val_df.to_csv(
    DAY4_DIR / "validation_split.csv",
    index=False
)

test_df.to_csv(
    DAY4_DIR / "test_split.csv",
    index=False
)

# Save combined split information
all_splits = []

for split_name, split_df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:

    temp = split_df.copy()
    temp["split"] = split_name
    all_splits.append(temp)

split_df = pd.concat(
    all_splits,
    ignore_index=True
)

split_df.to_csv(
    DAY4_DIR / "all_splits.csv",
    index=False
)

print("✓ Split files saved to:")
print(DAY4_DIR)

✓ Split files saved to:
/content/drive/MyDrive/Underwater-Image-Data-set-main/Day_4_Preprocessing


In [11]:
IMAGE_SIZE = 224

# ImageNet normalization
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    # Mild augmentation only
    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=MEAN,
        std=STD
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=MEAN,
        std=STD
    )
])

print("✓ Transforms created")
print("Image size:", IMAGE_SIZE)

✓ Transforms created
Image size: 224


In [12]:
class UnderwaterTripletDataset(Dataset):
    def __init__(self, dataframe, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        # Load the three corresponding images
        input_img = Image.open(row["input"]).convert("RGB")
        target_img = Image.open(row["target"]).convert("RGB")
        gen_img = Image.open(row["gen"]).convert("RGB")

        # Apply transformations
        if self.transform:

            input_img = self.transform(input_img)
            target_img = self.transform(target_img)
            gen_img = self.transform(gen_img)

        return {
            "input": input_img,
            "target": target_img,
            "generated": gen_img,
            "image_id": int(row["image_id"]),
            "source_archive": row["source_archive"]
        }


print("✓ Dataset class created")

✓ Dataset class created


In [13]:
BATCH_SIZE = 16
NUM_WORKERS = 2

train_dataset = UnderwaterTripletDataset(
    train_df,
    transform=train_transform
)

val_dataset = UnderwaterTripletDataset(
    val_df,
    transform=eval_transform
)

test_dataset = UnderwaterTripletDataset(
    test_df,
    transform=eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("✓ DataLoaders created")
print()
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

✓ DataLoaders created

Training batches: 14
Validation batches: 3
Test batches: 3


In [14]:
batch = next(iter(train_loader))

print("Input shape:")
print(batch["input"].shape)

print("\nTarget shape:")
print(batch["target"].shape)

print("\nGenerated shape:")
print(batch["generated"].shape)

print("\nImage IDs:")
print(batch["image_id"])

print("\n✓ DataLoader is working!")

Input shape:
torch.Size([16, 3, 224, 224])

Target shape:
torch.Size([16, 3, 224, 224])

Generated shape:
torch.Size([16, 3, 224, 224])

Image IDs:
tensor([ 11, 112,  19,   2,  12,  35,  45,  16,  26,  53,  15, 153,   7,  10,
         93,   1])

✓ DataLoader is working!


In [15]:
config = {
    "version": "v1.0",

    "image_size": IMAGE_SIZE,

    "normalization": {
        "mean": MEAN,
        "std": STD,
        "type": "ImageNet"
    },

    "augmentation": {
        "horizontal_flip": True,
        "horizontal_flip_probability": 0.5
    },

    "split": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
        "random_seed": SEED,
        "leakage_safe": True,
        "unit_of_split": "triplet/image_id"
    },

    "dataloader": {
        "batch_size": BATCH_SIZE,
        "shuffle_train": True,
        "shuffle_validation": False,
        "shuffle_test": False
    },

    "dataset": {
        "total_triplets": len(triplets),
        "train_triplets": len(train_df),
        "validation_triplets": len(val_df),
        "test_triplets": len(test_df)
    }
}

config_path = DAY4_DIR / "preprocessing_config_v1.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("✓ Configuration saved:")
print(config_path)

✓ Configuration saved:
/content/drive/MyDrive/Underwater-Image-Data-set-main/Day_4_Preprocessing/preprocessing_config_v1.json


In [16]:
summary = pd.DataFrame({
    "Property": [
        "Total triplets",
        "Training triplets",
        "Validation triplets",
        "Test triplets",
        "Image size",
        "Normalization",
        "Augmentation",
        "Random seed",
        "Leakage check"
    ],

    "Value": [
        len(triplets),
        len(train_df),
        len(val_df),
        len(test_df),
        f"{IMAGE_SIZE} x {IMAGE_SIZE}",
        "ImageNet mean/std",
        "Random horizontal flip (p=0.5)",
        SEED,
        "Passed"
    ]
})

summary.to_csv(
    DAY4_DIR / "preprocessing_summary_v1.csv",
    index=False
)

print(summary)
print("\n✓ Preprocessing summary saved.")

              Property                           Value
0       Total triplets                             314
1    Training triplets                             219
2  Validation triplets                              47
3        Test triplets                              48
4           Image size                       224 x 224
5        Normalization               ImageNet mean/std
6         Augmentation  Random horizontal flip (p=0.5)
7          Random seed                              42
8        Leakage check                          Passed

✓ Preprocessing summary saved.
